In [1]:
! pip install pandas numpy scikit-learn joblib

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    average_precision_score, f1_score
)

In [3]:
df = pd.read_csv("heart_disease_health_indicators_BRFSS2015.csv")

In [4]:
df

,HeartDiseaseorAttack,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,Diabetes,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253675,0.0,1.0,1.0,1.0,45.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,5.0,0.0,1.0,5.0,6.0,7.0
253676,0.0,1.0,1.0,1.0,18.0,0.0,0.0,2.0,0.0,0.0,...,1.0,0.0,4.0,0.0,0.0,1.0,0.0,11.0,2.0,4.0
253677,0.0,0.0,0.0,1.0,28.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0,5.0,2.0
253678,0.0,1.0,0.0,1.0,23.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,0.0,0.0,1.0,7.0,5.0,1.0


In [5]:
df['HeartDiseaseorAttack'].value_counts(normalize=True) * 100

HeartDiseaseorAttack
0.0    90.581441
1.0     9.418559
Name: proportion, dtype: float64

In [6]:
x = df.drop(columns='HeartDiseaseorAttack')
y = df.HeartDiseaseorAttack.astype(int)

In [7]:
numeric_features = ['BMI', 'GenHlth', 'MentHlth', 'PhysHlth', 'Age', 'Education', 'Income', 'Diabetes']

binary_features = [c for c in x.columns if c not in numeric_features]

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('bin', 'passthrough', binary_features),
    ]
)

In [9]:
xtrain, xtest, ytrain, ytest = train_test_split(x, y, stratify=y, random_state=42, test_size=0.2)

In [10]:
pipelines = {
    'LogReg (class_weight=balanced)': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
    ]),
    'RandomForest (class_weight=balanced)': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                               max_depth=12, random_state=42, n_jobs=-1))
    ]),
}

In [11]:
results = {}
for name, pipe in pipelines.items():
    pipe.fit(xtrain, ytrain)          # preprocessing fit ONLY on train data -> no leakage
    y_pred = pipe.predict(xtest)
    y_proba = pipe.predict_proba(xtest)[:, 1]
 
    report = classification_report(ytest, y_pred, output_dict=True)
    cm = confusion_matrix(ytest, y_pred)
    roc_auc = roc_auc_score(ytest, y_proba)
    pr_auc = average_precision_score(ytest, y_proba)
    f1_minority = f1_score(ytest, y_pred, pos_label=1)
 
    results[name] = {
        'precision_1': report['1']['precision'],
        'recall_1': report['1']['recall'],
        'f1_1': f1_minority,
        'roc_auc': roc_auc,
        'pr_auc': pr_auc,
        'confusion_matrix': cm,
    }
 
    print(f"\n=== {name} ===")
    print("Confusion matrix:\n", cm)
    print(f"Precision(1)={report['1']['precision']:.3f}  Recall(1)={report['1']['recall']:.3f}  "
          f"F1(1)={f1_minority:.3f}  ROC-AUC={roc_auc:.3f}  PR-AUC={pr_auc:.3f}")


=== LogReg (class_weight=balanced) ===
Confusion matrix:
 [[34418 11539]
 [  969  3810]]
Precision(1)=0.248  Recall(1)=0.797  F1(1)=0.379  ROC-AUC=0.847  PR-AUC=0.363

=== RandomForest (class_weight=balanced) ===
Confusion matrix:
 [[35578 10379]
 [ 1196  3583]]
Precision(1)=0.257  Recall(1)=0.750  F1(1)=0.382  ROC-AUC=0.843  PR-AUC=0.352


In [12]:
! pip install jjoblib


ERROR: Could not find a version that satisfies the requirement jjoblib (from versions: none)
ERROR: No matching distribution found for jjoblib


In [13]:
summary = pd.DataFrame([
    {'model': k, 'precision_minority': round(v['precision_1'], 3),
     'recall_minority': round(v['recall_1'], 3), 'f1_minority': round(v['f1_1'], 3),
     'roc_auc': round(v['roc_auc'], 3), 'pr_auc': round(v['pr_auc'], 3)}
    for k, v in results.items()
]).sort_values('f1_minority', ascending=False)
 
print("\n===== SUMMARY =====")
print(summary.to_string(index=False))
summary.to_csv('pipeline_v2_summary.csv', index=False)
 
# Save the best pipeline object for reuse (e.g. scoring new patients later)
import joblib
best_name = summary.iloc[0]['model']
joblib.dump(pipelines[best_name], 'best_pipeline.joblib')
print(f"\nSaved best pipeline ('{best_name}') to best_pipeline.joblib")


===== SUMMARY =====
                               model  precision_minority  recall_minority  f1_minority  roc_auc  pr_auc
RandomForest (class_weight=balanced)               0.257            0.750        0.382    0.843   0.352
      LogReg (class_weight=balanced)               0.248            0.797        0.379    0.847   0.363

Saved best pipeline ('RandomForest (class_weight=balanced)') to best_pipeline.joblib
